In [0]:
%run ../config

In [0]:
sales_df = read_data(path=bronze_layer['sales_data'],type='csv')\
            .select('date','store_nbr','family','sales','onpromotion')
sales_df.limit(5).display()


In [0]:
print(f"number of records : {sales_df.count()}")
print(f"number of unique stores {sales_df.select('store_nbr').distinct().count()}")
print(f"number of unique families {sales_df.select('family').distinct().count()}")
print(f"data is from {sales_df.agg(f.min('date')).collect()[0][0]} to {sales_df.agg(f.max('date')).collect()[0][0]}")

## Data Sanity checks

### S1.missing days

In [0]:

min_date = sales_df.agg(F.min('date')).collect()[0][0]
max_date = sales_df.agg(F.max('date')).collect()[0][0]
days = (max_date-min_date).days + 1
print(min_date, max_date,days)

sales_df.groupBy('store_nbr','family').agg(f.countDistinct('date').alias('date_c'),F.min('date'),F.max('date')).display()

In [0]:
full_dates_set = set([str(datetime.date(d)) for d in pd.date_range(start=min_date, end=max_date)])
prd_str_dates = set([str(d['date']) for d in sales_df.filter(f.col('store_nbr')==20).filter(f.col('family')=="CLEANING").select('date').distinct().collect()])
print(full_dates_set - prd_str_dates)

- seems like on christmas the stores are closed and the sales for these dates are not present in sales data.

### S2.negative sales

In [0]:
sales_df.filter(f.col('sales')<0).display()

- no negative sales